# Dim_Location - Gold

- Table: `pondolens.dev_gold.dim_location`
- Grain: one row per standardized location code.
- Sources: Silver GRDP per capita, regional appropriations, regional obligations, service access, and unemployment rate.
- Standardizes code casing, spaces, hyphens, and recognized aliases.
- Uses the first available name by source priority: GRDP, appropriations, obligations, service access, then unemployment.
- Geographic level: PHL and NW = nationwide; CO = other; remaining codes = region.
- Location IDs start at 10001. Adding codes can change generated IDs, so dependent fact-table keys must be rebuilt.

In [0]:
%sql

CREATE OR REPLACE TABLE pondolens.dev_gold.dim_location
USING DELTA
AS

WITH all_locations AS (
    SELECT
        region_code,
        region_name,
        1 AS source_priority
    FROM pondolens.dev_silver.grdp_per_capita

    UNION ALL

    SELECT
        region_code,
        region_name,
        2 AS source_priority
    FROM pondolens.dev_silver.regional_appropriations

    UNION ALL

    SELECT
        region_code,
        region_name,
        3 AS source_priority
    FROM pondolens.dev_silver.regional_obligations

    UNION ALL

    SELECT
        region_code,
        region AS region_name,
        4 AS source_priority
    FROM pondolens.dev_silver.service_access_available

    UNION ALL

    SELECT
        region_code,
        region_name,
        5 AS source_priority
    FROM pondolens.dev_silver.unemployment_rate
),

cleaned AS (
    SELECT
        REGEXP_REPLACE(
            UPPER(TRIM(region_code)),
            r'[\s\-–—]+',
            ''
        ) AS clean_code,
        NULLIF(
            TRIM(
                REGEXP_REPLACE(
                    UPPER(region_name),
                    r'\s+',
                    ' '
                )
            ),
            ''
        ) AS region,
        source_priority
    FROM all_locations
),

standardized AS (
    SELECT
        CASE
            WHEN clean_code IN ('PH', 'PHL', 'PHILIPPINES')
                THEN 'PHL'
            WHEN clean_code IN ('NW', 'NATIONWIDE')
                THEN 'NW'
            WHEN clean_code IN ('CO', 'CENTRALOFFICE')
                THEN 'CO'
            WHEN clean_code IN (
                'IVA', '4A', 'REGIONIVA', 'REGION4A', 'CALABARZON'
            )
                THEN 'IVA'
            WHEN clean_code IN (
                'IVB', '4B', 'REGIONIVB', 'REGION4B',
                'MIMAROPA', 'MIMAROPAREGION'
            )
                THEN 'IVB'
            WHEN clean_code = 'NATIONALCAPITALREGION'
                THEN 'NCR'
            WHEN clean_code = 'CORDILLERAADMINISTRATIVEREGION'
                THEN 'CAR'
            WHEN clean_code = 'NEGROSISLANDREGION'
                THEN 'NIR'
            WHEN clean_code =
                'BANGSAMOROAUTONOMOUSREGIONINMUSLIMMINDANAO'
                THEN 'BARMM'
            ELSE REGEXP_REPLACE(
                clean_code,
                r'^REGION(?=(XIII|XII|XI|IX|VIII|VII|VI|III|II|X|V|I)$)',
                ''
            )
        END AS region_code,
        region,
        source_priority
    FROM cleaned
    WHERE clean_code IS NOT NULL
      AND clean_code <> ''
),

deduplicated AS (
    SELECT
        region_code,
        region,
        ROW_NUMBER() OVER (
            PARTITION BY region_code
            ORDER BY
                CASE WHEN region IS NULL THEN 1 ELSE 0 END,
                source_priority,
                region
        ) AS rn
    FROM standardized
)

SELECT
    CAST(
        10000 + ROW_NUMBER() OVER (ORDER BY region_code)
        AS INT
    ) AS location_id,
    region_code,
    CASE
        WHEN region_code = 'PHL' THEN 'PHILIPPINES'
        WHEN region_code = 'NW' THEN 'NATIONWIDE'
        WHEN region_code = 'CO' THEN 'CENTRAL OFFICE'
        ELSE COALESCE(region, region_code)
    END AS region,
    CASE
        WHEN region_code IN ('PHL', 'NW') THEN 'nationwide'
        WHEN region_code = 'CO' THEN 'other'
        ELSE 'region'
    END AS geographic_level
FROM deduplicated
WHERE rn = 1;

SELECT
    location_id,
    region_code,
    region,
    geographic_level
FROM pondolens.dev_gold.dim_location
ORDER BY location_id;

In [0]:
%sql
SELECT *
FROM pondolens.dev_gold.dim_location;

## Data Quality Checks

- Checks required fields, duplicate keys, ID range, formatting, and geographic-level assignments.
- Duplicate checks count all rows sharing a duplicated ID or code.
- PASS = 0% affected; REVIEW = above 0% up to 5%; FAIL = above 5%.
- An empty table is FAIL. These checks report issues without stopping execution.

In [0]:
%sql

WITH base AS (
    SELECT
        *,
        COUNT(*) OVER (
            PARTITION BY location_id
        ) AS id_count,
        COUNT(*) OVER (
            PARTITION BY region_code
        ) AS code_count
    FROM pondolens.dev_gold.dim_location
),

checks AS (
    SELECT
        1 AS check_order,
        'Missing required fields' AS dq_check,
        COUNT(*) AS total_checked,
        COUNT(CASE
            WHEN location_id IS NULL
              OR NULLIF(TRIM(region_code), '') IS NULL
              OR NULLIF(TRIM(region), '') IS NULL
              OR NULLIF(TRIM(geographic_level), '') IS NULL
            THEN 1
        END) AS affected_count
    FROM base

    UNION ALL

    SELECT
        2,
        'Duplicate location IDs',
        COUNT(*),
        COUNT(CASE
            WHEN location_id IS NOT NULL AND id_count > 1
            THEN 1
        END)
    FROM base

    UNION ALL

    SELECT
        3,
        'Duplicate location codes',
        COUNT(*),
        COUNT(CASE
            WHEN region_code IS NOT NULL AND code_count > 1
            THEN 1
        END)
    FROM base

    UNION ALL

    SELECT
        4,
        'Location IDs below 10001',
        COUNT(*),
        COUNT(CASE
            WHEN location_id < 10001 THEN 1
        END)
    FROM base

    UNION ALL

    SELECT
        5,
        'Incorrect code formatting',
        COUNT(*),
        COUNT(CASE
            WHEN region_code != UPPER(TRIM(region_code))
              OR region_code RLIKE r'[\s\-–—]'
            THEN 1
        END)
    FROM base

    UNION ALL

    SELECT
        6,
        'Incorrect name formatting',
        COUNT(*),
        COUNT(CASE
            WHEN region != TRIM(
                REGEXP_REPLACE(UPPER(region), r'\s+', ' ')
            )
            THEN 1
        END)
    FROM base

    UNION ALL

    SELECT
        7,
        'Incorrect geographic level',
        COUNT(*),
        COUNT(CASE
            WHEN NOT (
                geographic_level <=> CASE
                    WHEN region_code IN ('PHL', 'NW')
                        THEN 'nationwide'
                    WHEN region_code = 'CO'
                        THEN 'other'
                    ELSE 'region'
                END
            )
            THEN 1
        END)
    FROM base

    UNION ALL

    SELECT
        8,
        'Incorrect special location names',
        COUNT(*),
        COUNT(CASE
            WHEN region_code IN ('PHL', 'NW', 'CO')
             AND NOT (
                region <=> CASE
                    WHEN region_code = 'PHL' THEN 'PHILIPPINES'
                    WHEN region_code = 'NW' THEN 'NATIONWIDE'
                    WHEN region_code = 'CO' THEN 'CENTRAL OFFICE'
                END
             )
            THEN 1
        END)
    FROM base
)

SELECT
    dq_check,
    total_checked,
    affected_count,
    ROUND(
        100.0 * affected_count / NULLIF(total_checked, 0),
        2
    ) AS affected_percentage,
    CASE
        WHEN total_checked = 0 THEN 'FAIL'
        WHEN affected_count = 0 THEN 'PASS'
        WHEN 100.0 * affected_count / total_checked <= 5
            THEN 'REVIEW'
        ELSE 'FAIL'
    END AS status
FROM checks
ORDER BY check_order;